# Social Media Engagement — Predictive Pipeline

**Business question:** How accurately can we predict `engagement_rate` from **actionable** post design features (no audience-scale inputs)?

**Modeling goal — Prediction (Ch. 11–12):** Train/test evaluation; compare **LinearRegression**, **DecisionTreeRegressor**, and **RandomForestRegressor** in a `Pipeline` with `ColumnTransformer`; export `models/social_engagement_predictor.pkl`.

**Prerequisite:** Run the **explanatory** notebook first so `social_engagement_insights.json` contains OLS factors; this notebook **merges** predictive metrics into that file.

**Feature parity:** Uses `get_x_y` from `ml_scripts/social_engagement_features.py` (same as `social_engagement_scorer.py`).


## 1. Load data & feature matrix


In [10]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

_cwd = Path.cwd().resolve()
_root = _cwd.parent if _cwd.name == "ml-pipeline" else _cwd
sys.path.insert(0, str(_root / "ml_scripts"))
from social_engagement_features import CATEGORICAL, get_x_y  # noqa: E402


def resolve_csv_path() -> Path:
    for p in [
        _root / "backend" / "AngelsLandingv2.API" / "SeedData" / "social_media_posts.csv",
        _root / "lighthouse_csv_v7" / "social_media_posts.csv",
        _root.parent / "lighthouse_csv_v7" / "social_media_posts.csv",
    ]:
        if p.is_file():
            return p
    raise FileNotFoundError("social_media_posts.csv not found")


df = pd.read_csv(resolve_csv_path())
X, y, post_ids = get_x_y(df)
print("Rows:", len(X), "features:", list(X.columns))


Rows: 812 features: ['platform', 'day_of_week', 'post_type', 'media_type', 'call_to_action_type', 'content_topic', 'sentiment_tone', 'post_hour', 'num_hashtags', 'mentions_count', 'caption_length', 'log_impressions', 'log_followers', 'has_call_to_action', 'features_resident_story', 'is_boosted']


## 2. Compare models (holdout)


In [11]:
# Ch. 11–12: preprocess + compare models on holdout
num_cols = [c for c in X.columns if c not in CATEGORICAL]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", StandardScaler(), num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipelines = {
    "LinearRegression": Pipeline([("prep", preprocess), ("m", LinearRegression())]),
    "DecisionTree": Pipeline([("prep", preprocess), ("m", DecisionTreeRegressor(max_depth=8, min_samples_leaf=3, random_state=42))]),
    "RandomForest": Pipeline([("prep", preprocess), ("m", RandomForestRegressor(n_estimators=160, max_depth=14, min_samples_leaf=3, random_state=42, n_jobs=-1))]),
}

results = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred),
    })
print(pd.DataFrame(results))

baseline_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"Baseline MAE (predict train mean): {baseline_mae:.5f}")


              model       MAE      RMSE        R2
0  LinearRegression  0.030260  0.037576  0.474927
1      DecisionTree  0.028382  0.038593  0.446106
2      RandomForest  0.022073  0.029742  0.671040
Baseline MAE (predict train mean): 0.04235


## 3. Export production model + merge metrics + post predictions JSON


In [12]:
# Ch. 17: export RandomForest — holdout metrics first, then refit on full data
rf_pipe = pipelines["RandomForest"]
pt = rf_pipe.predict(X_test)
insights_path = _root / "backend" / "AngelsLandingv2.API" / "SeedData" / "social_engagement_insights.json"
payload = {}
if insights_path.is_file():
    payload = json.loads(insights_path.read_text(encoding="utf-8"))
payload["predictiveMaeHoldout"] = float(mean_absolute_error(y_test, pt))
payload["predictiveR2Holdout"] = float(r2_score(y_test, pt))
payload["predictiveRmseHoldout"] = float(mean_squared_error(y_test, pt) ** 0.5)

rf_pipe.fit(X, y)
pred_all = rf_pipe.predict(X)

models_dir = _root / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "social_engagement_predictor.pkl"
joblib.dump(rf_pipe, model_path)
print("Saved:", model_path)

rf = rf_pipe.named_steps["m"]
prep = rf_pipe.named_steps["prep"]
imp = pd.Series(rf.feature_importances_, index=prep.get_feature_names_out()).sort_values(ascending=False)
insights_path.parent.mkdir(parents=True, exist_ok=True)
insights_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Updated insights JSON")

scored_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
pred_rows = [
    {"postId": int(pid), "predictedEngagementRate": float(pr), "engagementScoredAt": scored_at}
    for pid, pr in zip(post_ids.astype(int), pred_all)
]
pred_path = insights_path.parent / "social_engagement_post_predictions.json"
pred_path.write_text(json.dumps(pred_rows, indent=2), encoding="utf-8")
print("Wrote", pred_path)

imp.head(15)


Saved: /Users/zacharywaldrip/Intex2/AngelsLandingv2/models/social_engagement_predictor.pkl
Updated insights JSON
Wrote /Users/zacharywaldrip/Intex2/AngelsLandingv2/backend/AngelsLandingv2.API/SeedData/social_engagement_post_predictions.json


num__post_hour                        0.461952
cat__sentiment_tone_Informative       0.137706
num__log_impressions                  0.056746
cat__sentiment_tone_Grateful          0.047917
num__log_followers                    0.033494
cat__sentiment_tone_Hopeful           0.031679
num__caption_length                   0.030684
cat__call_to_action_type_(missing)    0.023655
num__has_call_to_action               0.019966
cat__sentiment_tone_Celebratory       0.015600
num__num_hashtags                     0.012337
cat__platform_TikTok                  0.009691
num__mentions_count                   0.007175
cat__sentiment_tone_Emotional         0.007028
cat__sentiment_tone_Urgent            0.006045
dtype: float64